# Tien leerbare schaalfactoren per MX-groep / A32
## SVHN · CNN-8 · honderd epochs · schaalkeuzes leren tot epoch 100

Iedere groep van **32 gewichten** kiest één factor uit **×1, ×2, ×4, ×8, ×16, ×32, ×64, ×128, ×256 en ×512**. De E4M3-bron wordt uit de FP32-master bepaald met dezelfde preprocessing en encoder als eerder. E2M3 behoudt de hoogste twee exponentbits; E2M1 ontstaat door twee mantissabits van E2M3 te verwijderen. Beide lagere modi delen **E8M0-broncode + gekozen offset 0..9**. De E4M3-bronschaal zelf wordt niet geleerd.

De **4.080 FP32-hulpwaarden** $\theta_b$ beginnen exact op **6**, dus aanvankelijk ×64, zonder extra willekeurige initialisatie. Hun bereik is $[0,9]$. De forward rondt af naar het dichtstbijzijnde gehele getal; halve waarden gaan naar de hogere offset. De benaderde schaalgradient is $\ln(2)\sum_{i\in b}g_iq_i$ per lagere modus. De twee bijdragen worden opgeteld. Na één gezamenlijke Adam-update worden de hulpwaarden teruggebracht binnen $[0,9]$. Activaties zijn **A32**, masters en optimizerstaat **FP32**.

Dit experiment begint **opnieuw vanaf nul** met dezelfde opzet als de geslaagde vijf-epochpilot. Gewichten én schaalhulpwaarden leren gedurende **alle honderd epochs**. De leersnelheid daalt voor beide groepen na epochs 50/75/90. Checkpoint50 wordt volledig bewaard, zodat een eventuele latere tak met bevroren offsets vanuit exact die staat kan vertrekken. Die tak wordt hier niet uitgevoerd.

Dit notebook leest alleen JSON/CSV. Het start geen training, opent geen dataset en laadt geen modelcheckpoint. De hoofdvergelijking gebruikt de echte eerdere **binaire honderd-epochtraining met leren tot 100** en de **vaste ×32/×64-trainingen**. De nieuwe vijf-epochpilot staat apart als vroege controle. De template wordt zonder uitgevoerde cellen geleverd. Voer zelf **Run All** uit.


In [ ]:
from pathlib import Path
RUN_DIR = None  # Automatisch de nieuwste training; of Path(r'C:\...\resultaatmap').
SAVE_FIGURES = False  # True schrijft naar een nieuwe analysis_<tijdstip>-map.


In [ ]:
import sys
import importlib.util
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown(str):
        pass
    def display(value):
        print(value)

# De helper werkt vanuit de pakketmap, projectroot, resultaatmap of uitgepakte feedbackmap.
roots = []
if RUN_DIR is not None and Path(RUN_DIR).expanduser().is_dir():
    roots.append(Path(RUN_DIR).expanduser().resolve())
roots += [Path.cwd(), *Path.cwd().parents]
candidates = []
for root in dict.fromkeys(roots):
    candidates += [root / 'notebook_support.py',
                   root / 'software_checks' / 'mx_triple_learned_scale10_reference100_v1' / 'notebook_support.py',
                   root / 'runner_source' / 'notebook_support.py']
helper_path = next((p for p in candidates if p.is_file()), None)
if helper_path is None:
    raise FileNotFoundError('Houd notebook_support.py bij de notebook of open vanuit de projectmap.')
helper_name = '_mx_triple_learned_scale10_reference100_analysis_helper'
spec = importlib.util.spec_from_file_location(helper_name, helper_path)
if spec is None or spec.loader is None:
    raise FileNotFoundError(f'Analysehelper ontbreekt: {helper_path}')
ns = importlib.util.module_from_spec(spec)
sys.modules[helper_name] = ns
spec.loader.exec_module(ns)
run = ns.load_run(RUN_DIR)
figures = {}
print('Ingelezen resultaatmap:', run.path)
display(Markdown(ns.status_markdown(run)))
display(ns.protocol_table(run))
display(Markdown('### Meldingen en ontbrekende gegevens'))
display(ns.diagnostics_table(run))


## Epoch100 en beste checkpoint gekozen door E4M3

Het beste checkpoint is het **eerste strikte maximum van de E4M3-testaccuracy binnen de gemeten training**. Een gelijke score vervangt het eerdere checkpoint niet. E2M3 en E2M1 worden op precies dezelfde epoch gerapporteerd; hun eigen hoogste scores worden niet samengevoegd met een andere E4M3-epoch.

De officiële testset wordt volgens het bestaande protocol voor checkpointselectie gebruikt. De beste score is daardoor geen onafhankelijke testschatting na selectie. Het vaste eindpunt **epoch 100** staat er apart naast zodra de volledige training technisch is bevestigd. Bij een onvoltooide run is het eerste paneel uitdrukkelijk de laatste afgeronde epoch; er wordt geen ontbrekende epoch 100-score ingevuld.


In [ ]:
display(ns.checkpoint_table(run).round({'Testaccuracy [%]': 6, 'Harde test-CE': 8}))
figures['checkpoint_accuracy'] = ns.plot_checkpoint_accuracy(run)
display(figures['checkpoint_accuracy'])
plt.close(figures['checkpoint_accuracy'])


## Vergelijking met twee leerbare keuzes en vaste ×32/×64

Alle hoofdreferenties zijn werkelijk uitgevoerde **honderd-epochtrainingen**. De binaire referentie leert ×32/×64 gedurende alle honderd epochs; zij heeft geen bevriezing op 50. Voor een onvoltooide nieuwe training gebruiken we alleen dezelfde reeds afgeronde epochrange. De meegeleverde bestanden en hun herkomst worden met hashes gecontroleerd.

De grafieken tonen het **volledige accuracyverloop**, de drie accuracies op **epoch 100** en op **het checkpoint gekozen door E4M3 binnen dezelfde gemeten epochrange**. Iedere training kiest haar eigen E4M3-bestepoch; haar drie precisies blijven samen. Er worden geen latere referentie-epochs gebruikt om een kortere run te beoordelen.

De leerbare varianten beginnen op dezelfde factor ×64. De tienkeuzevariant verruimt de toegestane beweging; we voegen geen exploratieruis, extra trainingspasses of extra FP32-schaalparameters toe. Eén run per variant laat geen uitspraak over statistische significantie toe.


In [ ]:
references = ns.load_fixed_references(run)
display(Markdown(ns.comparison_markdown(run, references)))
display(ns.comparison_table(run, references).round(6))
figures['scale_comparison_curves'] = ns.plot_scale_comparison_curves(run, references)
display(figures['scale_comparison_curves'])
plt.close(figures['scale_comparison_curves'])
figures['scale_comparison_checkpoints'] = ns.plot_scale_comparison_checkpoints(run, references)
display(figures['scale_comparison_checkpoints'])
plt.close(figures['scale_comparison_checkpoints'])


## Vroege controle tegenover de uitgevoerde tienkeuzepilot

De pilot is een aparte training met hetzelfde recept. De volledige run begint opnieuw bij epoch 0. Hieronder worden alleen de twee epoch 5-metingen vergeleken; het pilotcheckpoint is geen warmstart.


In [ ]:
pilot_reference = ns.load_pilot_reference(run)
display(Markdown(ns.pilot_comparison_markdown(run, pilot_reference)))
display(ns.pilot_comparison_table(run, pilot_reference).round(6))


## Train- en testaccuracy per modus

De drie modi worden gezamenlijk vanaf een nieuwe initialisatie getraind. Trainaccuracy wordt tijdens de updates verzameld. Testaccuracy wordt na iedere epoch gemeten met de opgeslagen harde blokkeuzes, bevroren evaluatiestaat en volledige testset. De drie modusgebonden BatchNorm-banken leren tijdens de training normaal mee. De E4M3-route blijft direct uit de actuele bron bepaald; E2M3 en E2M1 volgen dezelfde geleerde blokkeuze.


In [ ]:
figures['accuracy_curves'] = ns.plot_learning(run)
display(figures['accuracy_curves'])
plt.close(figures['accuracy_curves'])


## Losses: trainingsdoelen onderscheiden van harde test-CE

De trainpass van E4M3 gebruikt harde CE van labels. Daarna leert E2M3 met soft-target CE van de **detached E4M3-softmax**; E2M1 leert met soft-target CE van de **detached E2M3-softmax**. Dit is recursieve distillatie met T=1. De drie gradients worden gesommeerd zonder deling door drie, waarna één Adam-update volgt; er is geen extra FP32-teacherforward.

De drie trainobjectieven staan in afzonderlijke panelen omdat hun targets verschillen. Soft CE is geen harde label-CE en wordt hier ook niet als KL-divergentie aangeduid. **Testloss is voor alle drie modi harde CE van dezelfde labels** en staat daarom samen in één vergelijkbaar paneel.


In [ ]:
if not run.metrics.empty:
    objective_columns = [key for key in ('epoch', 'phase', 'weight_format', 'objective_type', 'objective_loss') if key in run.metrics]
    display(run.metrics[objective_columns].round({'objective_loss': 8}))
figures['objective_losses'] = ns.plot_objectives(run)
display(figures['objective_losses'])
plt.close(figures['objective_losses'])


## Blijft het netwerk verschillende cijfers voorspellen?

De eerdere post-hocproeven lieten bij sommige routes sterk vertekende of constante voorspellingen zien. Daarom wordt hier per testepoch en per modus bijgehouden hoe de voorspellingen over de tien cijfers verdeeld zijn. Een constante voorspelling is direct zichtbaar als **één gebruikte klasse en 100% concentratie**. De tabel toont ook hoe vaak het dominante cijfer werkelijk in de testset voorkomt.

Dit zijn geaggregeerde tellingen; deze notebook doet geen nieuwe inferentie en leidt er geen per-beeldvergelijking uit af.


In [ ]:
display(ns.prediction_table(run, selected_only=True))
figures['prediction_diversity'] = ns.plot_prediction_diversity(run)
display(figures['prediction_diversity'])
plt.close(figures['prediction_diversity'])


## Welke factoren worden gekozen, en bewegen de hulpwaarden?

De diagnostiek onderscheidt het leersignaal, de continue beweging van θ en de harde keuze. De negen drempels liggen op **0,5; 1,5; …; 8,5**. Een parameter kan bewegen zonder een drempel te passeren. Geen harde omschakeling is daarom geen technische FAIL.

De gestapelde grafieken tonen **alle tien factoren per laag en per epoch**, inclusief epoch 0. De histogrammen tonen de volledige θ-verdeling bij initialisatie, bij het E4M3-bestcheckpoint en bij de laatste afgeronde epoch. We onderscheiden groepen die een uiterste factor kiezen van hulpwaarden die exact op de projectiegrens 0 of 9 liggen.

De helper berekent keuzes en samenvattingen opnieuw uit de opgeslagen blokstaat en vergelijkt ze met CSV en history. De training hoeft niet alle toegestane factoren daadwerkelijk te bezoeken; de afzonderlijke codeccontroles toetsen hun bereikbaarheid.


In [ ]:
display(Markdown(ns.scale_learning_markdown(run)))
display(ns.scale_table(run).round(8))
figures['scale_choices'] = ns.plot_scale_choices(run)
display(figures['scale_choices'])
plt.close(figures['scale_choices'])
figures['scale_boundaries'] = ns.plot_scale_boundaries(run)
display(figures['scale_boundaries'])
plt.close(figures['scale_boundaries'])


In [ ]:
figures['theta_distributions'] = ns.plot_theta_distributions(run)
display(figures['theta_distributions'])
plt.close(figures['theta_distributions'])


### Wissels, spronggrootte, gradients en projectie

Iedere verandering van de harde offset na een optimizerupdate telt als één wissel. Een sprong van +5 naar +7 telt dus als **één wissel met afstand twee**. De tabel bewaart ook de som van sprongafstanden en de grootste sprong. Het aantal unieke wisselende groepen telt elke groep maximaal eenmaal per epoch; netto wijzigingen vergelijken alleen de epochgrenzen. Een groep kan meerdere keren heen en weer wisselen.

De L2-update is het netto verschil van de continue parameters. De gradientgrafiek toont de grootste waargenomen gradientnorm per laag en epoch. De projectietelling telt waarden die na Adam buiten [0,9] vielen en werden teruggezet, inclusief herhalingen. Deze diagnostieken zijn geen bewijs van accuracywinst.


In [ ]:
figures['scale_switches'] = ns.plot_scale_switches(run)
display(figures['scale_switches'])
plt.close(figures['scale_switches'])


In [ ]:
figures['scale_gradients'] = ns.plot_scale_gradients(run)
display(figures['scale_gradients'])
plt.close(figures['scale_gradients'])


In [ ]:
display(Markdown('## Gemeten trainingsduur; bij onvolledige run een afzonderlijke projectie'))
display(Markdown(ns.timing_markdown(run)))
figures['timing'] = ns.plot_timing(run)
display(figures['timing'])
plt.close(figures['timing'])


In [ ]:
display(Markdown('## GPU-geheugen\n\nDe opgeslagen pieken betreffen de PyTorch CUDA-allocator. Zij meten niet alle GPU-processen of compacte native MX-uitvoering. Ontbrekende waarden blijven ontbrekend.'))
figures['gpu_memory'] = ns.plot_memory(run)
display(figures['gpu_memory'])
plt.close(figures['gpu_memory'])


In [ ]:
display(Markdown('## Opgeslagen technische controles\n\nDeze tabellen rapporteren bestaande controles uit de run. Het notebook voert geen nieuwe gradient-, model- of checkpointcontrole uit.'))
display(ns.audit_table(run))
display(Markdown('### Gewichtsupdates\n\nDe mastergewichten zijn dezelfde gedeelde tensors. De herhaalde masterkolommen per modus mogen **niet bij elkaar worden opgeteld**; code- en schaalwijzigingen behoren wel bij het betreffende formaat.'))
display(ns.weight_update_table(run))
display(Markdown('### Eén opgeslagen E4M3-bron en twee afgeleide exports per geselecteerd checkpoint'))
display(ns.export_table(run))


## Opslagkosten van meer keuzemogelijkheden

Een compacte flexibele E4-bron bewaart nu **4 bits per groep** voor de gekozen offset. Tien mogelijkheden passen in een nibble; waarden 10–15 zijn ongeldig. Voor 4.080 groepen zijn dit **2.040 bytes**, tegenover **510 bytes** voor de binaire keuze: **1.530 bytes extra**. Een losse lagere uitvoerprecisie heeft de gekozen offset al verwerkt in zijn E8M0-schaal; daar wordt de keuze-index niet nogmaals bijgeteld.

Er blijft één FP32-parameter per groep. Parameter, gradient en twee FP32-Adam-momenten vragen analytisch **65.280 extra bytes** ten opzichte van vaste schalen, evenveel als bij de binaire leerproef. Tijdelijke buffers, activaties, optimizerstaptellers en allocatoroverhead zijn niet inbegrepen. De grafiek toont deze onderdelen afzonderlijk, niet totale VRAM of de grootte van `.pt`-bestanden.


In [ ]:
display(ns.storage_cost_table(run))
figures['storage_cost'] = ns.plot_storage_cost(run)
display(figures['storage_cost'])
plt.close(figures['storage_cost'])


## Interpretatie en grenzen

Tegenover de binaire leerproef zijn de **tien gehele offsets 0..9 in plaats van twee offsets 5/6** en de ruimere parametergrenzen de inhoudelijke wijziging. De groepindeling, oorspronkelijke modelinitialisatie, θ-init 6, losses, drie passes, schaalgradient en optimizerinstellingen zijn behouden. Tegenover de tienkeuzepilot verandert alleen de duur: deze run begint opnieuw en loopt door tot 100. De E4-bronschalen blijven dynamisch volgens dezelfde encoder; de gewichtsgradient blijft identity-STE naar G(W).

Meer keuzes vergroten het bereik; zij voegen geen tussenwaarden tussen ×32 en ×64 toe. Een lage accuracy, constante voorspelling of weinig harde wissels is een onderzoeksuitkomst wanneer de technische controles slagen. Bekijk accuracy, harde test-CE, voorspeldistributie en schaalgedrag samen. Dit experiment bewijst niet welk bereik, welke gradientbenadering of welk bevriestijdstip optimaal is.

**Er wordt in deze run niets bevroren op epoch 50.** Het checkpoint van50 bewaart model, θ, Adam, scheduler, RNG en historie. Een eventuele latere vriesproef kan daarmee vanaf 51 starten zonder deze tak met verder leren opnieuw te trainen. Onafhankelijke schalen per exponentbreedte blijven een afzonderlijke volgende onderzoeksrichting.

Bij honderd volledig bevestigde epochs wordt de gemeten som van epochtijden getoond; bij een onvoltooide run blijft een projectie duidelijk afzonderlijk. CUDA volgt de oorspronkelijke backend en biedt geen bitexacte herhaal- of hervatgarantie. De FP32-emulator meet geen native MX-hardwaredoorvoer of energie. De officiële testset wordt ook voor checkpointselectie gebruikt; de geselecteerde score is geen onafhankelijke eindtestschatting.


In [ ]:
if SAVE_FIGURES:
    analysis_dir = ns.save_figures(figures, run)
    print('Figuren opgeslagen in:', analysis_dir)
else:
    print('Alleen analyse in het notebook; geen figuren naar schijf geschreven.')
print('Ingelezen resultaatmap:', run.path)
display(pd.DataFrame({'Ingelezen bestand': [str(path.relative_to(run.path)) if path.is_relative_to(run.path) else str(path) for path in run.input_files]}))
